### ray actor 创建

In [1]:
import gc
import json
import time
import subprocess
import sys
import tempfile
import textwrap
from pprint import pprint
import ray
import requests

SANDBOX_URL = "http://10.1.123.37:8080/run_code"

def reset_ray(namespace: str):
    if ray.is_initialized():
        ray.shutdown()
    return ray.init(namespace=namespace, ignore_reinit_error=True, log_to_driver=False)

def call_run_code(code: str, *, sandbox_url: str = SANDBOX_URL, run_timeout: int = 10, memory_limit_mb: int = 256):
    payload = {
        "compile_timeout": run_timeout,
        "run_timeout": run_timeout,
        "code": code,
        "stdin": "",
        "memory_limit_MB": memory_limit_mb,
        "language": "python",
        "files": {},
        "fetch_files": [],
    }
    resp = requests.post(
        sandbox_url,
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        data=json.dumps(payload),
        timeout=run_timeout + 10,
    )
    resp.raise_for_status()
    return resp.json()

@ray.remote
class ExecutionWorker:
    def __init__(self, sandbox_url: str):
        self.sandbox_url = sandbox_url

    def ping(self):
        return "pong"

    def execute_python(self, code: str, run_timeout: int = 10):
        return call_run_code(code, sandbox_url=self.sandbox_url, run_timeout=run_timeout)

class SandboxTool:
    def __init__(self, actor_name: str, *, detached: bool, sandbox_url: str = SANDBOX_URL):
        self.actor_name = actor_name
        options = {"name": actor_name, "get_if_exists": True}
        if detached:
            options["lifetime"] = "detached"
        self.execution_pool = ExecutionWorker.options(**options).remote(sandbox_url)

    def execute(self, code: str):
        return ray.get(self.execution_pool.execute_python.remote(code))

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-23 15:45:51,032	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


### Owner 与 Unowned Handle (非拥有句柄)

在 Ray 中，Actor 的生命周期是由 **Owner（拥有者）** 决定的：
1. **Owner**：第一次调用 `.remote()` 创建这个 Actor 的进程/变量。只要 Owner 的句柄还在，Actor 就活着。（默认 attached lifetime）
2. **Unowned Handle (非拥有句柄)**：其他地方通过 `get_if_exists=True` 或 `ray.get_actor()` 拿到的句柄。

**致命规则**：非拥有句柄 **不能** 阻止 Actor 被回收！只要 Owner 丢弃了原始句柄，Ray 就会立刻杀死 Actor，此时所有拿着非拥有句柄的人都会报错 `ActorDiedError`。

下面这段代码，完美模拟了 `verl` 中 `RLHFDataset` (Owner) 和 `ToolAgentLoop` (Worker) 之间的冲突：

In [2]:
reset_ray("tutorial-unowned-handle")

print("【阶段 1】Driver (模拟 RLHFDataset) 初始化，创建了 Actor，成为 Owner")
driver_tool = SandboxTool(actor_name="shared-pool", detached=False)
print("   -> Driver 句柄可用:", ray.get(driver_tool.execution_pool.ping.remote()))

print("\n【阶段 2】Worker (模拟 ToolAgentLoop) 发现同名 Actor，获取了 Unowned Handle (非拥有句柄)")
# get_if_exists=True 会返回已存在 Actor 的句柄，但不会转移 Owner 身份
worker_tool = SandboxTool(actor_name="shared-pool", detached=False)
print("   -> Worker 句柄可用:", ray.get(worker_tool.execution_pool.ping.remote()))

print("\n【阶段 3】Driver 初始化结束，局部变量被回收，Owner 丢弃了原始句柄")
del driver_tool
gc.collect()  # 强制 Python 垃圾回收
time.sleep(1) # 等待 Ray 异步杀死 Actor

print("\n【阶段 4】Worker 尝试在 Rollout 时使用 Sandbox")
try:
    ray.get(worker_tool.execution_pool.ping.remote())
except Exception as e:
    print("   -> Worker 踩雷报错！异常类型:", type(e).__name__)
    print("   -> 错误信息:", str(e).split('\n')[0]) # 只打印第一行核心报错

ray.shutdown()

2026-04-23 15:46:05,485	INFO worker.py:2012 -- Started a local Ray instance.
/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


【阶段 1】Driver (模拟 RLHFDataset) 初始化，创建了 Actor，成为 Owner
   -> Driver 句柄可用: pong

【阶段 2】Worker (模拟 ToolAgentLoop) 发现同名 Actor，获取了 Unowned Handle (非拥有句柄)
   -> Worker 句柄可用: pong

【阶段 3】Driver 初始化结束，局部变量被回收，Owner 丢弃了原始句柄

【阶段 4】Worker 尝试在 Rollout 时使用 Sandbox
   -> Worker 踩雷报错！异常类型: ActorDiedError
   -> 错误信息: The actor died unexpectedly before finishing this task.


### 多进程下的 Owner 死亡

在真实的 `verl` 训练中，Driver 和 Rollout Worker 往往运行在**不同的进程**中。
如果 Owner 所在的进程直接退出了，效果是一样的：Actor 会因为 Owner 死亡而被连坐处决。

下面的代码启动了两个子进程：
- `Child 1` 是 Owner，创建 Actor 后退出。
- `Child 2` 是 Worker，拿到句柄后不断调用。
- 观察 `Child 1` 退出时，`Child 2` 会发生什么。

In [8]:
reset_ray("tutorial-owner-process-death")
ray_address = ray.get_runtime_context().gcs_address

child1_script = textwrap.dedent(f'''
    import time, ray
    ray.init(address={ray_address!r}, namespace="tutorial-owner-process-death", log_to_driver=False)
    
    @ray.remote
    class OwnerActor:
        def ping(self): return "pong"
        
    actor = OwnerActor.options(name="process-demo-actor", get_if_exists=True).remote()
    print("Child 1 (Owner) 创建了 Actor:", ray.get(actor.ping.remote()), flush=True)
    time.sleep(2.0)
    print("Child 1 (Owner) 进程退出！", flush=True)
    ray.shutdown()
''')

child2_script = textwrap.dedent(f'''
    import time, ray
    ray.init(address={ray_address!r}, namespace="tutorial-owner-process-death", log_to_driver=False)
    time.sleep(0.5)
    
    actor = ray.get_actor("process-demo-actor")
    print("Child 2 (Worker) 拿到了 Unowned Handle", flush=True)

    for i in range(10):
        try:
            ray.get(actor.ping.remote(), timeout=0.5)
            print(f"Child 2 ping {{i}}: 成功", flush=True)
        except Exception as e:
            print(f"Child 2 在 iter {{i}} 崩溃！类型: {{type(e).__name__}}", flush=True)
            print(f"错误原因: {{str(e).split('The actor is dead because')[1].strip()}}", flush=True)
            break
        time.sleep(0.4)
    ray.shutdown()
''')

with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f1, \
     tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f2:
    f1.write(child1_script)
    f2.write(child2_script)
    p1, p2 = f1.name, f2.name

proc1 = subprocess.Popen([sys.executable, p1], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
proc2 = subprocess.Popen([sys.executable, p2], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

out1, _ = proc1.communicate(timeout=30)
out2, _ = proc2.communicate(timeout=30)

print("=== Child 1 (Owner) 输出 ===\n" + out1.strip())
print("\n=== Child 2 (Worker) 输出 ===\n" + out2.strip())

ray.shutdown()

2026-04-23 15:55:45,890	INFO worker.py:2012 -- Started a local Ray instance.


=== Child 1 (Owner) 输出 ===
2026-04-23 15:55:47,765	INFO worker.py:1832 -- Connecting to existing Ray cluster at address: 10.2.160.10:54577...
2026-04-23 15:55:47,779	INFO worker.py:2012 -- Connected to Ray cluster.
/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
Child 1 (Owner) 创建了 Actor: pong
Child 1 (Owner) 进程退出！

=== Child 2 (Worker) 输出 ===
2026-04-23 15:55:47,781	INFO worker.py:1832 -- Connecting to existing Ray cluster at address: 10.2.160.10:54577...
2026-04-23 15:55:47,794	INFO worker.py:2012 -- Connected to Ray cluster.
/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray

### `lifetime="detached"`

既然问题出在 Owner 绑定上，解法就很简单了：**让 Actor 独立存活 (Detached)**。

`lifetime="detached"` 告诉 Ray：“不要把这个 Actor 的命和创建它的 Owner 绑在一起。即使 Owner 丢了句柄或者退出了，也要让它作为全局服务一直活着。”

我们用第一节的单进程代码再跑一次，仅仅把 `detached` 设为 `True`：

In [4]:
reset_ray("tutorial-detached-fix")

print("【阶段 1】Driver 初始化，创建 Detached Actor")
driver_tool = SandboxTool(actor_name="fixed-pool", detached=True)
print("   -> Driver 句柄可用:", ray.get(driver_tool.execution_pool.ping.remote()))

print("\n【阶段 2】Worker 获取 Unowned Handle")
worker_tool = SandboxTool(actor_name="fixed-pool", detached=True)
print("   -> Worker 句柄可用:", ray.get(worker_tool.execution_pool.ping.remote()))

print("\n【阶段 3】Driver 丢弃原始句柄")
del driver_tool
gc.collect()
time.sleep(1)

print("\n【阶段 4】Worker 尝试使用 Sandbox")
try:
    print("   -> Worker 依然可以成功调用！返回:", ray.get(worker_tool.execution_pool.ping.remote()))
    # 顺便测一下真实的远端代码执行
    result = worker_tool.execute("print('Detached Actor 完美运行！')")
    print("   -> Sandbox 执行结果:", result["run_result"]["stdout"].strip())
except Exception as e:
    print("   -> 报错了:", e)

# 清理 detached actor
try:
    ray.kill(ray.get_actor("fixed-pool"), no_restart=True)
except:
    pass
ray.shutdown()

2026-04-23 15:54:09,351	INFO worker.py:2012 -- Started a local Ray instance.


【阶段 1】Driver 初始化，创建 Detached Actor
   -> Driver 句柄可用: pong

【阶段 2】Worker 获取 Unowned Handle
   -> Worker 句柄可用: pong

【阶段 3】Driver 丢弃原始句柄

【阶段 4】Worker 尝试使用 Sandbox
   -> Worker 依然可以成功调用！返回: pong
   -> Sandbox 执行结果: Detached Actor 完美运行！
